# Imports

In [19]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
import ipywidgets as widgets
from ipywidgets import interact

# Data loading & understanding

In [20]:
# 1. Prepare the Data
iris = load_iris()
X = iris.data
feature_names = iris.feature_names

In [21]:
import pandas as pd

# Create a DataFrame from the original features
df = pd.DataFrame(X, columns=feature_names)
df['species'] = iris.target_names[iris.target]

In [22]:
print("Dataset shape:", X.shape)
print("\nFirst 10 rows of the Iris dataset:")

display(df.iloc[:10])
display(df.iloc[50:60])
display(df.iloc[100:110])

Dataset shape: (150, 4)

First 10 rows of the Iris dataset:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
5,5.4,3.9,1.7,0.4,setosa
6,4.6,3.4,1.4,0.3,setosa
7,5.0,3.4,1.5,0.2,setosa
8,4.4,2.9,1.4,0.2,setosa
9,4.9,3.1,1.5,0.1,setosa


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
50,7.0,3.2,4.7,1.4,versicolor
51,6.4,3.2,4.5,1.5,versicolor
52,6.9,3.1,4.9,1.5,versicolor
53,5.5,2.3,4.0,1.3,versicolor
54,6.5,2.8,4.6,1.5,versicolor
55,5.7,2.8,4.5,1.3,versicolor
56,6.3,3.3,4.7,1.6,versicolor
57,4.9,2.4,3.3,1.0,versicolor
58,6.6,2.9,4.6,1.3,versicolor
59,5.2,2.7,3.9,1.4,versicolor


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),species
100,6.3,3.3,6.0,2.5,virginica
101,5.8,2.7,5.1,1.9,virginica
102,7.1,3.0,5.9,2.1,virginica
103,6.3,2.9,5.6,1.8,virginica
104,6.5,3.0,5.8,2.2,virginica
105,7.6,3.0,6.6,2.1,virginica
106,4.9,2.5,4.5,1.7,virginica
107,7.3,2.9,6.3,1.8,virginica
108,6.7,2.5,5.8,1.8,virginica
109,7.2,3.6,6.1,2.5,virginica


# PCA

In [3]:
# Get PCA components for comparison
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# PCA in detail

In [11]:
X_pca

array([[-2.68412563,  0.31939725],
       [-2.71414169, -0.17700123],
       [-2.88899057, -0.14494943],
       [-2.74534286, -0.31829898],
       [-2.72871654,  0.32675451],
       [-2.28085963,  0.74133045],
       [-2.82053775, -0.08946138],
       [-2.62614497,  0.16338496],
       [-2.88638273, -0.57831175],
       [-2.6727558 , -0.11377425],
       [-2.50694709,  0.6450689 ],
       [-2.61275523,  0.01472994],
       [-2.78610927, -0.235112  ],
       [-3.22380374, -0.51139459],
       [-2.64475039,  1.17876464],
       [-2.38603903,  1.33806233],
       [-2.62352788,  0.81067951],
       [-2.64829671,  0.31184914],
       [-2.19982032,  0.87283904],
       [-2.5879864 ,  0.51356031],
       [-2.31025622,  0.39134594],
       [-2.54370523,  0.43299606],
       [-3.21593942,  0.13346807],
       [-2.30273318,  0.09870885],
       [-2.35575405, -0.03728186],
       [-2.50666891, -0.14601688],
       [-2.46882007,  0.13095149],
       [-2.56231991,  0.36771886],
       [-2.63953472,

In [4]:
# We'll use Petal Length (index 2) and Petal Width (index 3) for the raw 2D plot
# because they offer the best visual separation in the original feature space.
X_raw_2d = X[:, 2:4] 
raw_x_label = feature_names[2]
raw_y_label = feature_names[3]

In [5]:
def plot_clustering(algorithm, k, eps, min_samples):
    """
    Dynamically applies the chosen algorithm to both raw and PCA data,
    rendering a side-by-side comparison.
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Configure the chosen algorithm
    if algorithm == 'K-Means':
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        title_suffix = f"K-Means (k={k})"
    else:
        model = DBSCAN(eps=eps, min_samples=min_samples)
        title_suffix = f"DBSCAN (eps={eps:.2f}, minPts={min_samples})"
        
    # --- PLOT 1: RAW FEATURES (No PCA) ---
    labels_raw = model.fit_predict(X_raw_2d)
    
    # Plotting raw features
    scatter1 = ax1.scatter(X_raw_2d[:, 0], X_raw_2d[:, 1], c=labels_raw, 
                           cmap='tab10', edgecolor='k', s=60, alpha=0.8)
    
    # Highlight noise if DBSCAN
    if algorithm == 'DBSCAN' and -1 in labels_raw:
        noise_mask = (labels_raw == -1)
        ax1.scatter(X_raw_2d[noise_mask, 0], X_raw_2d[noise_mask, 1], 
                    c='black', marker='x', s=40, label='Noise')
        ax1.legend()
        
    ax1.set_title(f"Original Features: {title_suffix}", fontsize=12)
    ax1.set_xlabel(raw_x_label)
    ax1.set_ylabel(raw_y_label)
    ax1.grid(True, linestyle='--', alpha=0.6)

    # --- PLOT 2: PCA REDUCED ---
    labels_pca = model.fit_predict(X_pca)
    
    # Plotting PCA features
    scatter2 = ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_pca, 
                           cmap='tab10', edgecolor='k', s=60, alpha=0.8)
    
    if algorithm == 'DBSCAN' and -1 in labels_pca:
        noise_mask = (labels_pca == -1)
        ax2.scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], 
                    c='black', marker='x', s=40, label='Noise')
        ax2.legend()
        
    ax2.set_title(f"PCA Reduced: {title_suffix}", fontsize=12)
    ax2.set_xlabel('Principal Component 1')
    ax2.set_ylabel('Principal Component 2')
    ax2.grid(True, linestyle='--', alpha=0.6)

    plt.suptitle("Clustering Mechanics: Physical Reality vs. Mathematical Projection", fontsize=16)
    plt.tight_layout()
    plt.show()

In [9]:
# Generate pair plot
sns.pairplot(df, hue='species')
plt.show()

In [10]:
plt.figure(figsize=(8, 6))
for i, species in enumerate(iris.target_names):
    plt.scatter(X_pca[iris.target == i, 0], X_pca[iris.target == i, 1], label=species)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('PCA of Iris Dataset with Original Labels')
plt.legend()
plt.show()

In [6]:
# 2. Setup the Interactive UI
interact(plot_clustering, 
         algorithm=widgets.Dropdown(options=['K-Means', 'DBSCAN'], value='K-Means', description='Algorithm:'),
         k=widgets.IntSlider(min=2, max=6, step=1, value=3, description='K (K-Means):'),
         eps=widgets.FloatSlider(min=0.1, max=1.5, step=0.1, value=0.5, description='eps (DBSCAN):'),
         min_samples=widgets.IntSlider(min=2, max=15, step=1, value=5, description='minPts (DBSCAN):'));

interactive(children=(Dropdown(description='Algorithm:', options=('K-Means', 'DBSCAN'), value='K-Means'), IntS…